# Lab 9 - Association Rule Mining



Association rule mining started off as market basket analysis, which looked at items customers bought frequently together. Through association rule mining, we can discover interesting associations that help business decision-making processes such as catalog design, cross-marketing, customer shopping behavior analysis and more.

Basic Principle:

- **Frequent patterns** are itemsets that appear frequently in a dataset (e.g. transaction records).
- Items that are frequently associated (e.g. purchased) together can be represented as **association rules**
- **Support** and **Confidence** are measures of rule interestingness

**Support** is essentially the percentage of orders/transactions containing the itemset, given by the following formula:

$$\mathrm{Support}(A \Rightarrow B)=\dfrac{|A \cup B|}{n}$$

Given two item sets, A and B, **Confidence** measures the percentage of times that B was purchased, given that A was purchased. In other words, it measures how *confident* we are that B would follow A. This is calculated using the following formula:

$$\mathrm{Confidence}(A \Rightarrow B)=\dfrac{\mathrm{Support}(A \cup B)}{\mathrm{Support}(A)}$$

## Import Libraries and Load Dataset

In [1]:
# Uncomment and execute this cell if running locally or on Colab
# to install the mlxtend package for assoc. rule mining

#!pip install mlxtend --upgrade

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


%matplotlib inline
plt.style.use('fivethirtyeight')
sns.set_style('whitegrid')

In [3]:
 try:
    df = pd.read_csv('../data/groceries_dataset.csv')
except:
    df = pd.read_csv('https://raw.githubusercontent.com/GUC-DM/W2025/refs/heads/main/data/groceries_dataset.csv')
df.head()

HTTPError: HTTP Error 404: Not Found

In [ ]:
df.info()

## Exploratoy Data Analysis & Data Pre-processing

Converting the date column to a `datetime` datatype allows us to access useful datetime properties like day and month later on.

In this simple example we passed the `dayfirst` parameter for `pd.to_datetime` to correctly parse the date. Other parameters like `utc` can also be passed, in order to have the transaction time in addition to the date. This should always be handled in realistic applications, since you'll have to consider the data's date format (e.g., MM/DD or DD/MM), timezone, business days,...

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df.head()

In [ ]:
df.info()

Printing out the different varieties of products being sold along with the products themselves allows us to better understand the data we're dealing with. For actual retail stores with potentially thousands of different products, we could go for random sampling instead.

In [ ]:
print("There are {} products:".format(df['itemDescription'].nunique()))
df['itemDescription'].unique()

A few visualizations to better understand the business and the products that it sells.

In [ ]:
ax = df['itemDescription'].value_counts().nlargest(15).plot(kind='bar')
ax.set_title('Most Frequently Purchased Items', size=20)
ax.set_ylabel('Count');

In [ ]:
ax = df['itemDescription'].value_counts().nsmallest(15).plot(kind='bar')
ax.set_title('Least Sold Items', size=20)
ax.set_ylabel('Count');

Since the `'Date'` column is of type `datetime`, we can group the data by each month. It's possible to also group the data by weekday, or even by hour (if we had the transaction time in addition to the date).

In [ ]:
ax = df['Member_number'].groupby([df['Date'].dt.month]).nunique().plot(kind='bar')
ax.set_title('Unique Customers per Month', size=20)
ax.set_ylabel('Count');

To build association rules for the products being sold, we'll need to transform our data to a form where each row represents a single 'basket' or order, and each product is represented as a one-hot encoded column. To address the first issue (and partially, the second issue), we'll need to apply the opposite of `melt` (lab 4), which is called `pivot`.

The transactions in this data are split over multiple rows, the primary key for a single order being both `'Member_number'` and `'Date'`. We'll also drop any duplicate items in a single purchase, since we're only interested if the item appears or not and also because `pivot` will raise an error if there any duplicates.

**Note**: if each row was already complete and had the columns we are interested in, then we could use `pd.get_dummies` for the columns we're interested in finding associations for to one-hot encode the categorical data and convert the binary columns (e.g., Yes/No) to 0s and 1s.

In [ ]:
enc_df = df[['Member_number', 'Date', 'itemDescription']].drop_duplicates().pivot(index=['Member_number', 'Date'], columns='itemDescription', values='itemDescription')
enc_df

Since `NaN` represents items that were *not* bought in the transaction, and non-`NaN` represents items that *were* purchased, we can use `notnull()` to get our data in the final form necessary for using it in the association rule algorithm.

In [ ]:
enc_df = enc_df.notnull()
enc_df

With each row now representing a single 'basket' of items, we can perform some additional visualizations.

In [ ]:
basket_sizes = enc_df.sum(axis=1)
ax = basket_sizes.value_counts().plot.bar()
ax.set_title('Distribution of Basket Sizes', size=20)
ax.set_ylabel('Count')
ax.set_xlabel('Number of items in a single transaction.');

In [ ]:
single_items = enc_df[basket_sizes == 1].sum()
ax = single_items.nlargest(15).plot.bar()
ax.set_title('Items Commonly Bought Alone', size=20)
ax.set_ylabel('Number of times bought alone');

## Association Rule Mining

Support is essentially the percentage of orders/transactions containing the itemset, given by the following formula:

$$\mathrm{Support}(A \Rightarrow B)=\dfrac{|A \cup B|}{n}$$

The first step in association rule mining is getting the frequent item sets in the data. We don’t want to set the standard high, otherwise the number of possible association rules is limited. However, keep in mind that the lower this threshold is set, the more spurious and non-frequent associations are extracted, which is usually undesirable. Usually, we can start with a high `min_support` threshold like 0.5 (50%), and reduce it till we get a useful amount of itemsets.

In [ ]:
#uncomment the below code if you are working on a local repo, prob the older version (Install mlxtend)
#!pip install mlxtend
from mlxtend.frequent_patterns import apriori, association_rules

freq_items = apriori(enc_df, min_support=0.005, use_colnames=True)
freq_items

Given two item sets, A and B, confidence measures the percentage of times that B was purchased, given that A was purchased. In other words, it measures how *confident* we are that B would follow A. This is calculated using the following formula:

$$\mathrm{Confidence}(A \Rightarrow B)=\dfrac{\mathrm{Support}(A \cup B)}{\mathrm{Support}(A)}$$

The next step is building the association rules out of the frequent item sets we got from the last step. There are several metrics we can use to filter the association rules. We'll try confidence first. Again, we can start with a high value like 0.75 (75%) and reduce it till we can get interesting association rules.

In [ ]:
association_rules(freq_items, metric='confidence', min_threshold=0.1, num_itemsets = len(enc_df)).sort_values('confidence', ascending=False)

Since whole milk is the most commonly purchased item, it also appears as the consequent in many of the association rules. One useful association we can conclude from here is frankfurter and other vegetables. Though it appears that confidence is low for many of the itemsets; this is likely due to this store/dataset having a very small number of items per order (see the basket size visual above).

# Extra Reading

### Lift Metric

We can consider using lift, which, on the other hand, indicates if there is a relationship between itemsets A and B. The value can be interpreted as such:

- lift = 1 implies no relationship between A and B (i.e., A and B occur together only by chance)

- lift > 1 implies that there is a positive relationship between A and B. (i.e., A and B occur together more often than random)

- lift < 1 implies that there is a negative relationship between A and B. (i.e., A and B occur together less often than random)

$$\mathrm{Lift}(A, B) = \mathrm{Lift}(B, A) = \dfrac{\mathrm{Support}(A \cup B)}{\mathrm{Support}(A) \times \mathrm{Support}(B)}$$

In [ ]:
association_rules(freq_items, metric='lift', min_threshold=1, num_itemsets = len(enc_df)).sort_values('lift', ascending=False)

Buying one of frankfurter or other vegetables *lifts* the chance of the other being purchased. The same goes for sausage and yogurt, as well as sausage and soda.